In [1]:
import sys
import numpy
import transformers
from importlib.metadata import version
from transformers import RTDetrV2Config
from docling.document_converter import DocumentConverter

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Transformers:", transformers.__version__)
print("Docling:", version("docling"))
print("RT-DETRv2 support: OK")

/home/nicolas/anaconda3/envs/docling311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
NumPy: 2.4.6
Transformers: 5.14.1
Docling: 2.114.0
RT-DETRv2 support: OK


In [7]:
from docling.document_converter import DocumentConverter

source = "vw_test_set/annual-report-2025-volkswagen-group_page_0003.pdf"  # a document via a local path or URL
converter = DocumentConverter()
result = converter.convert(source)
print(result.document.export_to_markdown())  # output: "## Docling Technical Report[...]"

[INFO] 2026-07-22 17:34:22,411 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-22 17:34:22,413 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-22 17:34:22,465 [RapidOCR] download_file.py:60: File exists and is valid: /home/nicolas/anaconda3/envs/docling311/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-22 17:34:22,465 [RapidOCR] main.py:50: Using /home/nicolas/anaconda3/envs/docling311/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-22 17:34:22,903 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-22 17:34:22,904 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-22 17:34:22,907 [RapidOCR] download_file.py:60: File exists and is valid: /home/nicolas/anaconda3/envs/docling311/lib/python3.11/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-22 17:34:22,907 [RapidOCR] main.py:50: Using /home/nicolas/anaconda3/envs/docling311

<!-- image -->

## To our Shareholders

|   05 | Letter to our Shareholders            |
|------|---------------------------------------|
|   09 | The Board of Management of Volkswagen |
|      | Aktiengesellschaft                    |
|   11 | Report of the Supervisory Board       |

<!-- image -->

## Corporate Governance

Group Corporate Governance Declaration Members of the Board of Management Members of the Supervisory Board and Composition of the Committees Remuneration Report

<!-- image -->

## Group Management R e p or t

|   72 | Goals and Strategies                                                     |
|------|--------------------------------------------------------------------------|
|   78 | Internal Management System and                                           |
|   81 | Structure and Business Activities                                        |
|   85 | Disclosures Required under Takeover Law                                  |
|   88 | Business Development              

### First pass: page routing

In [8]:
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    HeadingHierarchyOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

pipeline_options = PdfPipelineOptions(
    # Expensive stages disabled
    do_ocr=False,
    do_table_structure=False,
    do_formula_enrichment=False,
    do_code_enrichment=False,
    do_picture_classification=False,
    do_picture_description=False,
    do_chart_extraction=False,

    # Do not create or retain images
    generate_page_images=False,
    generate_picture_images=False,
    generate_parsed_pages=False,

    # Prefer the PDF's embedded text layer
    force_backend_text=True,

    # Optional lightweight heading inference
    heading_hierarchy_options=HeadingHierarchyOptions(
        enabled=True,
        use_bookmarks=True,
        use_numbering=True,
        use_style=False,  # style inference requires parsed pages
    ),
)

converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF],
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    },
)

In [10]:
from collections import defaultdict
from pathlib import Path
import gc

PDF_PATH = Path("Annual-Report-2025-Bank-Vaduz-en.pdf")
TOTAL_PAGES = 52
BLOCK_SIZE = 30

page_lines: dict[int, list[str]] = defaultdict(list)
page_labels: dict[int, list[tuple[str, str]]] = defaultdict(list)

for start_page in range(1, TOTAL_PAGES + 1, BLOCK_SIZE):
    end_page = min(start_page + BLOCK_SIZE - 1, TOTAL_PAGES)

    print(f"Processing pages {start_page}-{end_page}")

    result = converter.convert(
        PDF_PATH,
        page_range=(start_page, end_page),
        raises_on_error=False,
    )

    for item, level in result.document.iterate_items():
        text = getattr(item, "text", None)
        provenance = getattr(item, "prov", None)

        if not text or not provenance:
            continue

        cleaned_text = " ".join(text.split())
        if not cleaned_text:
            continue

        label = str(getattr(item, "label", "unknown"))
        pages = {prov.page_no for prov in provenance}

        for page_number in pages:
            page_lines[page_number].append(cleaned_text)
            page_labels[page_number].append((label, cleaned_text))

    del result
    gc.collect()

Processing pages 1-30
Processing pages 31-52


## Page index

In [11]:
import pandas as pd

records = []

for page_number in range(1, TOTAL_PAGES + 1):
    lines = page_lines.get(page_number, [])
    text = "\n".join(lines)

    records.append(
        {
            "page": page_number,
            "character_count": len(text),
            "line_count": len(lines),
            "opening_lines": " | ".join(lines[:8]),
            "preview": text[:800],
        }
    )

page_index = pd.DataFrame(records)
page_index.to_csv("page_index_raw.csv", index=False)

print(page_index.head())

   page  character_count  line_count  \
0     1              129           2   
1     2              167           2   
2     3              470          16   
3     4              629          19   
4     5             1095         113   

                                       opening_lines  \
0  Annual Report 2025 | LGT Bank Ltd., Vaduz LGT ...   
1  LGT Bank Ltd., Vaduz LGT Bank Ltd. continued t...   
2  Contents | 4 Organisational structure | 5 The ...   
3  Organisational structure | December 2025 | Boa...   
4  The financial year in comparison | Balance she...   

                                             preview  
0  Annual Report 2025\nLGT Bank Ltd., Vaduz LGT B...  
1  LGT Bank Ltd., Vaduz LGT Bank Ltd. continued t...  
2  Contents\n4 Organisational structure\n5 The fi...  
3  Organisational structure\nDecember 2025\nBoard...  
4  The financial year in comparison\nBalance shee...  


In [13]:
pd.set_option('display.max_colwidth', None)

page_index.iloc[0]

page                                                                                                                                                 1
character_count                                                                                                                                    129
line_count                                                                                                                                           2
opening_lines      Annual Report 2025 | LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner.
preview             Annual Report 2025\nLGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner.
Name: 0, dtype: object

In [21]:
page_index.iloc[20]

page                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  21
character_count                                                                                

## Find headers

In [22]:
import re
import pandas as pd
from IPython.display import display

# ============================================================
# Configuration
# ============================================================

# Column labels and other fragments that should not be treated
# as section headers.
COMMON_TABLE_LABELS = {
    "book value",
    "carrying amount",
    "cost",
    "market value",
    "fair value",
    "total",
    "subtotal",
    "assets",
    "liabilities",
    "equity",
    "current assets",
    "non-current assets",
    "current liabilities",
    "non-current liabilities",
    "mortgage-backed",
    "mortgage backed",
    "other collateral",
    "without collateral",
    "previous year",
    "reporting year",
    "notes",
}

# Words that often indicate a major report section.
# These are supporting signals, not mandatory requirements.
SECTION_TERMS = {
    "balance sheet",
    "income statement",
    "cash flow",
    "financial statements",
    "notes to the financial statements",
    "accounting principles",
    "accounting policies",
    "risk management",
    "segment reporting",
    "off-balance sheet",
    "details on",
    "information on",
    "explanations",
    "consolidated",
}

# A proposed section header must reach this score.
SECTION_SCORE_THRESHOLD = 4

# Number of opening segments shown in the validation table.
OPENING_SEGMENTS_TO_DISPLAY = 8


# ============================================================
# Text helpers
# ============================================================

def normalize_text(value) -> str:
    """Normalize whitespace without changing the visible wording."""
    if value is None or pd.isna(value):
        return ""

    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_for_comparison(value) -> str:
    """More aggressive normalization for comparisons."""
    text = normalize_text(value).lower()
    text = text.replace("–", "-").replace("—", "-")
    return text.strip(" |:-")


def split_opening_lines(value) -> list[str]:
    """
    Split the opening_lines field on '|'.

    Example:
        'Details on the balance sheet | 1 Overview of collateral'
    becomes:
        ['Details on the balance sheet', '1 Overview of collateral']
    """
    text = normalize_text(value)

    if not text:
        return []

    return [
        normalize_text(segment)
        for segment in text.split("|")
        if normalize_text(segment)
    ]


def letter_ratio(text: str) -> float:
    """Return the share of non-space characters that are letters."""
    characters = [
        character
        for character in text
        if not character.isspace()
    ]

    if not characters:
        return 0.0

    letter_count = sum(character.isalpha() for character in characters)
    return letter_count / len(characters)


def numeric_ratio(text: str) -> float:
    """Return the share of non-space characters that are digits."""
    characters = [
        character
        for character in text
        if not character.isspace()
    ]

    if not characters:
        return 0.0

    digit_count = sum(character.isdigit() for character in characters)
    return digit_count / len(characters)


# ============================================================
# Patterns
# ============================================================

NUMBER_ONLY_RE = re.compile(r"^\d+$")

# Examples:
# "1 Overview of collateral"
# "3.2 Financial investments"
# "4.1.2 Credit risks"
NUMBERED_TITLE_RE = re.compile(
    r"^\d+(?:\.\d+)*[.)]?\s+\D.*$"
)

DATE_RE = re.compile(
    r"^(?:"
    r"\d{1,2}[./-]\d{1,2}[./-]\d{2,4}"
    r"|"
    r"\d{4}"
    r")$"
)

FINANCIAL_DATE_RE = re.compile(
    r"^\d{1,2}[./-]\d{1,2}[./-]\d{2,4}$"
)

UNIT_RE = re.compile(
    r"^\(?\s*"
    r"(?:CHF|EUR|USD|GBP|AUD|CAD|JPY)?"
    r"\s*"
    r"(?:in\s+)?"
    r"(?:units?|thousands?|millions?|billions?)"
    r"\s*\)?$",
    flags=re.IGNORECASE,
)

CURRENCY_UNIT_RE = re.compile(
    r"^\(?\s*(?:CHF|EUR|USD|GBP|AUD|CAD|JPY)"
    r"(?:\s+(?:thousands?|millions?|billions?))?"
    r"\s*\)?$",
    flags=re.IGNORECASE,
)


# ============================================================
# Fragment classification
# ============================================================

def looks_like_date(text: str) -> bool:
    return bool(DATE_RE.fullmatch(normalize_text(text)))


def looks_like_unit(text: str) -> bool:
    text = normalize_text(text)

    return bool(
        UNIT_RE.fullmatch(text)
        or CURRENCY_UNIT_RE.fullmatch(text)
    )


def looks_like_numbered_topic(text: str) -> bool:
    """Detect a number and title contained in the same segment."""
    return bool(NUMBERED_TITLE_RE.match(normalize_text(text)))


def looks_like_table_label(text: str) -> bool:
    """
    Identify common table-column labels, dates, units, and values.
    """
    normalized = normalize_for_comparison(text)

    if not normalized:
        return True

    if normalized in COMMON_TABLE_LABELS:
        return True

    if looks_like_date(text):
        return True

    if looks_like_unit(text):
        return True

    if NUMBER_ONLY_RE.fullmatch(normalized):
        return True

    # Mostly numeric strings are likely values, not headings.
    if numeric_ratio(normalized) >= 0.45:
        return True

    # Short percentage or financial-value fragments.
    if re.fullmatch(r"[()\-+.,'\s\d%]+", normalized):
        return True

    return False


def looks_like_heading_text(text: str) -> bool:
    """
    Decide whether a fragment has the general shape of a heading.

    This does not determine whether it is a major section heading;
    it only filters out obvious dates, values, units, and columns.
    """
    text = normalize_text(text)
    normalized = normalize_for_comparison(text)

    if not text:
        return False

    if looks_like_table_label(text):
        return False

    if looks_like_numbered_topic(text):
        return False

    if len(text) < 6 or len(text) > 180:
        return False

    word_count = len(text.split())

    if word_count < 2 or word_count > 22:
        return False

    if letter_ratio(text) < 0.60:
        return False

    # Avoid fragments ending in isolated table punctuation.
    if normalized.endswith((",", ";", ":")):
        return False

    return True


# ============================================================
# Page-topic extraction
# ============================================================

def extract_page_topic(segments: list[str]) -> str | None:
    """
    Extract a local numbered topic/table title.

    Handles:
        ['1 Overview of collateral (CHF thousands)', ...]
        ['3', 'Securities and precious metals held as', ...]
    """
    if not segments:
        return None

    first = segments[0]

    # Case 1:
    # "1 Overview of collateral (CHF thousands)"
    if looks_like_numbered_topic(first):
        return first

    # Case 2:
    # "3" | "Securities and precious metals held as" | ...
    if (
        NUMBER_ONLY_RE.fullmatch(first)
        and len(segments) >= 2
        and looks_like_heading_text(segments[1])
    ):
        topic_parts = [first, segments[1]]

        # Add one extra wrapped title segment when it looks safe.
        # Stop as soon as a table-column label, date, or unit begins.
        for segment in segments[2:4]:
            if looks_like_table_label(segment):
                break

            if looks_like_heading_text(segment):
                topic_parts.append(segment)
            else:
                break

        return " ".join(topic_parts)

    return None


# ============================================================
# Section-header scoring
# ============================================================

def section_header_score(
    segments: list[str],
    line_count: int | None = None,
    character_count: int | None = None,
) -> tuple[int, list[str]]:
    """
    Score the first opening segment as a possible section header.

    The strongest pattern in your report is:

        Section header
        |
        Numbered table/subsection title

    Example:
        Details on the balance sheet
        |
        1 Overview of collateral (CHF thousands)
    """
    if not segments:
        return 0, ["no opening segments"]

    first = segments[0]
    first_normalized = normalize_for_comparison(first)

    score = 0
    reasons = []

    # Basic heading-like shape.
    if looks_like_heading_text(first):
        score += 2
        reasons.append("first segment is heading-like")
    else:
        return 0, ["first segment is not heading-like"]

    # Strong signal: next segment contains a numbered title.
    if len(segments) >= 2 and looks_like_numbered_topic(segments[1]):
        score += 3
        reasons.append("followed by a numbered topic")

    # Alternative form:
    # section | 1 | title
    elif (
        len(segments) >= 3
        and NUMBER_ONLY_RE.fullmatch(segments[1])
        and looks_like_heading_text(segments[2])
    ):
        score += 3
        reasons.append("followed by a separate number and topic")

    # Section vocabulary is supporting evidence.
    if any(term in first_normalized for term in SECTION_TERMS):
        score += 2
        reasons.append("contains section vocabulary")

    # A heading should not be highly numeric.
    if numeric_ratio(first) == 0:
        score += 1
        reasons.append("contains no digits")

    # Very short labels can be ambiguous.
    if len(first.split()) <= 2:
        score -= 1
        reasons.append("very short heading candidate")

    # Dense table pages are common in financial reports.
    # We apply only a minor penalty because a section may begin above a table.
    if line_count is not None and line_count > 140:
        score -= 1
        reasons.append("page is table-dense")

    return score, reasons


def find_explicit_section_header(
    segments: list[str],
    line_count: int | None = None,
    character_count: int | None = None,
) -> tuple[str | None, int, list[str]]:
    """Return the explicit section header when the score is sufficient."""
    score, reasons = section_header_score(
        segments=segments,
        line_count=line_count,
        character_count=character_count,
    )

    if score >= SECTION_SCORE_THRESHOLD:
        return segments[0], score, reasons

    return None, score, reasons


# ============================================================
# Main extraction and propagation
# ============================================================

def extract_headers_per_page(
    page_index: pd.DataFrame,
) -> pd.DataFrame:
    """
    Extract explicit headers and inherit them on continuation pages.

    Required input columns:
        page
        opening_lines

    Optional columns:
        line_count
        character_count
        preview
    """
    required_columns = {"page", "opening_lines"}
    missing_columns = required_columns - set(page_index.columns)

    if missing_columns:
        raise ValueError(
            f"page_index is missing columns: {sorted(missing_columns)}"
        )

    working = (
        page_index
        .copy()
        .sort_values("page")
        .reset_index(drop=True)
    )

    current_section = None
    current_section_start_page = None

    extracted_records = []

    for _, row in working.iterrows():
        page_number = int(row["page"])
        segments = split_opening_lines(row["opening_lines"])

        line_count = row.get("line_count", None)
        character_count = row.get("character_count", None)

        explicit_header, header_score, score_reasons = (
            find_explicit_section_header(
                segments=segments,
                line_count=line_count,
                character_count=character_count,
            )
        )

        page_topic = extract_page_topic(segments)

        if explicit_header is not None:
            current_section = explicit_header
            current_section_start_page = page_number
            section_status = "explicit"
            inherited = False

        elif current_section is not None:
            section_status = "inherited"
            inherited = True

        else:
            section_status = "unknown"
            inherited = False

        extracted_records.append(
            {
                "page": page_number,

                # Header found directly on this page, if any.
                "explicit_header": explicit_header,

                # Effective header after forward propagation.
                "section_header": current_section,

                "section_status": section_status,
                "section_inherited": inherited,
                "section_start_page": current_section_start_page,

                # Local table/subsection subject.
                "page_topic": page_topic,

                # Useful for reviewing the heuristic.
                "header_score": header_score,
                "header_score_reasons": "; ".join(score_reasons),

                # Parsed opening-line information.
                "first_opening_segment": (
                    segments[0] if segments else None
                ),
                "opening_segments": " | ".join(
                    segments[:OPENING_SEGMENTS_TO_DISPLAY]
                ),

                # Original diagnostic information.
                "line_count": row.get("line_count", None),
                "character_count": row.get("character_count", None),
                "preview": row.get("preview", None),
            }
        )

    return pd.DataFrame(extracted_records)


# ============================================================
# Run extraction
# ============================================================

headers_per_page = extract_headers_per_page(page_index)


# ============================================================
# Display 1: every page for validation
# ============================================================

validation_columns = [
    "page",
    "section_status",
    "section_header",
    "section_start_page",
    "page_topic",
    "header_score",
    "first_opening_segment",
    "opening_segments",
]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)

print("HEADERS ASSIGNED TO EVERY PAGE")
display(
    headers_per_page[validation_columns]
    .style
    .set_properties(
        subset=[
            "section_header",
            "page_topic",
            "first_opening_segment",
            "opening_segments",
        ],
        **{"text-align": "left"}
    )
)


# ============================================================
# Display 2: only explicit section transitions
# ============================================================

explicit_transitions = headers_per_page.loc[
    headers_per_page["section_status"] == "explicit",
    [
        "page",
        "explicit_header",
        "header_score",
        "header_score_reasons",
        "page_topic",
        "opening_segments",
    ],
].reset_index(drop=True)

print("\nEXPLICIT SECTION TRANSITIONS")
display(explicit_transitions)


# ============================================================
# Display 3: suspicious or uncertain pages
# ============================================================

# These pages are useful to inspect manually:
# - no inherited or explicit section
# - close to the decision threshold
# - first segment looks heading-like but was rejected
suspicious_pages = headers_per_page.loc[
    (headers_per_page["section_status"] == "unknown")
    |
    (
        headers_per_page["header_score"]
        .between(
            max(1, SECTION_SCORE_THRESHOLD - 2),
            SECTION_SCORE_THRESHOLD - 1,
        )
    ),
    [
        "page",
        "section_status",
        "section_header",
        "header_score",
        "header_score_reasons",
        "first_opening_segment",
        "opening_segments",
    ],
].reset_index(drop=True)

print("\nUNKNOWN OR BORDERLINE PAGES")
display(suspicious_pages)


# ============================================================
# Save for review
# ============================================================

headers_per_page.to_csv(
    "headers_per_page_validation.csv",
    index=False,
)

explicit_transitions.to_csv(
    "explicit_section_transitions.csv",
    index=False,
)

print(
    "\nSaved:\n"
    "  - headers_per_page_validation.csv\n"
    "  - explicit_section_transitions.csv"
)

HEADERS ASSIGNED TO EVERY PAGE


,page,section_status,section_header,section_start_page,page_topic,header_score,first_opening_segment,opening_segments
0,1,unknown,nan,nan,nan,2,Annual Report 2025,"Annual Report 2025 | LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner."
1,2,unknown,nan,nan,nan,3,"LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner.","LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner. | Olivier de Perregaux, Chairman of the Board of Directors"
2,3,unknown,nan,nan,nan,0,Contents,Contents | 4 Organisational structure | 5 The financial year in comparison | 6 Message from the Chairman and CEO | 8 Balance sheet | 9 Off-balance sheet transactions | 10 Income statement | 11 Appropriation of profit
3,4,unknown,nan,nan,nan,2,Organisational structure,"Organisational structure | December 2025 | Board of Directors | Olivier de Perregaux, Chairman Gabrielle Nater-Bass Michael Bürge Roland Schubert Stephan Tanner Hans Roth | Internal Audit | Roman Berlinger | Executive Board | Roland Matt, CEO Ivo Klein Markus Werner Florian Dürselen Stefan F. Öhri"
4,5,unknown,nan,nan,nan,3,The financial year in comparison,The financial year in comparison | Balance sheet | 2025 | 2024 | Change | Absolute | % | Balance sheet total
5,6,unknown,nan,nan,nan,3,Message from the Chairman and CEO,"Message from the Chairman and CEO | Overall, global financial markets performed robustly in 2025, despite periods of heightened volatility amid ongoing geopolitical tensions and economic uncertainty. Against this backdrop, LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner – effectively leveraging its investment expertise across asset classes to deliver value to its clients. | Growth in core business | Net interest income decreased nearly 18 % year-on-year to CHF 343.5 million, as central banks lowered benchmark interest rates in 2025. In wealth management for private clients, LGT Bank Ltd.'s core business, commission income rose more than 11 % to CHF 480.5 million. | Business expenses edged 2 % lower from the prior year to CHF 778.6 million. Personnel expenses rose by CHF 4.8 million, or 1%, due to higher variable compensation. At the end of 2025, the number of employees stood at 1246 in full-time equivalents (previous year: 1312). | Overall, LGT Bank Ltd. recorded gross operating income of CHF 1067.5 million in 2025, or CHF 8.8 million lower on the year. Gross profit stood at CHF 289.0 million compared with CHF 280.6 million in 2024, resulting in a profit for the year of CHF 226.8 million, from CHF 200.5 million previously. The loan portfolio's excellent quality is reflected in the consistently low valuation adjustments required and in 2025, LGT Bank Ltd. recorded write-downs in line with those of the previous year. The balance sheet total decreased nearly 10 % to CHF 43.89 billion on the year, due mainly to foreign exchange movements. | Assets and capitalisation | Client assets of LGT Bank Ltd. rose nearly 9 % to CHF 142.2 billion, due primarily to net asset inflows of CHF 4.6 billion and market fluctuations including foreign exchange movements."
6,7,unknown,nan,nan,nan,0,nan,
7,8,unknown,nan,nan,nan,3,Balance sheet,Balance sheet | Assets (CHF thousands) | Note | 31.12.2025 | 31.12.2024 | Change | Absolute | %
8,9,explicit,Off-balance sheet transactions,9.000000,nan,5,Off-balance sheet transactions,Off-balance sheet transactions | Off-balance sheet (CHF thousands) | Note | 31.12.2025 | 31.12.2024 | Change | Absolute | %
9,10,inherited,Off-balance sheet transactions,9.000000,nan,3,Income statement,Income statement | Income statement (CHF thousands) | Note | 2025 | 2024 | Change | Absolute | %



EXPLICIT SECTION TRANSITIONS


,page,explicit_header,header_score,header_score_reasons,page_topic,opening_segments
0,9,Off-balance sheet transactions,5,first segment is heading-like; contains section vocabulary; contains no digits,NaN,Off-balance sheet transactions | Off-balance sheet (CHF thousands) | Note | 31.12.2025 | 31.12.2024 | Change | Absolute | %
1,15,Accounting principles,4,first segment is heading-like; contains section vocabulary; contains no digits; very short heading candidate,NaN,Accounting principles | Basic principles | The annual accounts are prepared in accordance with the act and ordinance on banks and invest...
2,20,Details on the balance sheet,7,first segment is heading-like; followed by a numbered topic; contains section vocabulary; contains no digits; page is table-dense,NaN,Details on the balance sheet | 1 Overview of collateral (CHF thousands) | Mortgage- | backed | Other | collateral | Without | collateral
3,34,Breakdown of balance sheet according to currencies,4,first segment is heading-like; contains section vocabulary; contains no digits; page is table-dense,NaN,Breakdown of balance sheet according to currencies | (CHF thousands) | CHF | EUR | USD | Other | 31.12.2024 | Total
4,36,Details on the off-balance sheet transactions,8,first segment is heading-like; followed by a numbered topic; contains section vocabulary; contains no digits,NaN,Details on the off-balance sheet transactions | 27 Contingent liabilities (CHF thousands) | 31.12.2025 | 31.12.2024 | Credit guarantees ...
5,38,Details on the income statement,8,first segment is heading-like; followed by a numbered topic; contains section vocabulary; contains no digits,NaN,Details on the income statement | 31 Offsetting of refinancing expenses with income from trading | The refinancing expenses arising from...
6,40,Additional information,5,first segment is heading-like; followed by a separate number and topic; contains no digits; very short heading candidate,NaN,Additional information | 40 | Securities negotiable on the stock exchange (CHF thousands) | 31.12.2025 | 31.12.2024 | Bonds and other fi...
7,46,"For further information on provisions for other business risks, refer to the following pages of the notes to the financial statements:",5,first segment is heading-like; contains section vocabulary; contains no digits,NaN,"For further information on provisions for other business risks, refer to the following pages of the notes to the financial statements: |..."



UNKNOWN OR BORDERLINE PAGES


,page,section_status,section_header,header_score,header_score_reasons,first_opening_segment,opening_segments
0,1,unknown,NaN,2,first segment is heading-like,Annual Report 2025,"Annual Report 2025 | LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner."
1,2,unknown,NaN,3,first segment is heading-like; contains no digits,"LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner.","LGT Bank Ltd., Vaduz LGT Bank Ltd. continued to execute its long-term growth strategy in a disciplined manner. | Olivier de Perregaux, C..."
2,3,unknown,NaN,0,first segment is not heading-like,Contents,Contents | 4 Organisational structure | 5 The financial year in comparison | 6 Message from the Chairman and CEO | 8 Balance sheet | 9 O...
3,4,unknown,NaN,2,first segment is heading-like; contains no digits; very short heading candidate,Organisational structure,"Organisational structure | December 2025 | Board of Directors | Olivier de Perregaux, Chairman Gabrielle Nater-Bass Michael Bürge Roland..."
4,5,unknown,NaN,3,first segment is heading-like; contains no digits,The financial year in comparison,The financial year in comparison | Balance sheet | 2025 | 2024 | Change | Absolute | % | Balance sheet total
5,6,unknown,NaN,3,first segment is heading-like; contains no digits,Message from the Chairman and CEO,"Message from the Chairman and CEO | Overall, global financial markets performed robustly in 2025, despite periods of heightened volatili..."
6,7,unknown,NaN,0,no opening segments,NaN,
7,8,unknown,NaN,3,first segment is heading-like; contains section vocabulary; contains no digits; very short heading candidate; page is table-dense,Balance sheet,Balance sheet | Assets (CHF thousands) | Note | 31.12.2025 | 31.12.2024 | Change | Absolute | %
8,10,inherited,Off-balance sheet transactions,3,first segment is heading-like; contains section vocabulary; contains no digits; very short heading candidate; page is table-dense,Income statement,Income statement | Income statement (CHF thousands) | Note | 2025 | 2024 | Change | Absolute | %
9,11,inherited,Off-balance sheet transactions,3,first segment is heading-like; contains no digits,Appropriation of profit,Appropriation of profit | Appropriation of profit – proposal of the Board of Directors | to the general meeting of shareholders (CHF tho...



Saved:
  - headers_per_page_validation.csv
  - explicit_section_transitions.csv


In [16]:
header_counts

Counter({'31.12.': 19,
         'change': 4,
         'other': 3,
         '(chf thousands)': 3,
         'counterparty': 3,
         'type(s) of transaction': 3,
         'lgt bank ltd., vaduz lgt bank ltd. continued to execute its long-term growth strategy in a disciplined manner.': 2,
         'balance sheet': 2,
         'assets': 2,
         'cash and cash equivalents': 2,
         'due from banks': 2,
         'bonds and other fixed interest-bearing securities': 2,
         'annual report': 1,
         'olivier de perregaux, chairman of the board of directors': 1,
         'contents': 1,
         '4 organisational structure': 1,
         '5 the financial year in comparison': 1,
         '6 message from the chairman and ceo': 1,
         '8 balance sheet': 1,
         '9 off-balance sheet transactions': 1,
         'organisational structure': 1,
         'december': 1,
         'board of directors': 1,
         'olivier de perregaux, chairman gabrielle nater-bass michael bürge rol